In this question, the input format is not particularly clear... so what i did was use the input from 
question 3 (multi-line fasta to single line) and did a global analysis(instead of fragments of sequence lines in the input)

<h3>This is using 1st order Markov Model</h3>

In [10]:
import os

def load_fasta_sequences(input_path):
    full_sequence = ""
    
    with open(input_path, 'r') as infile:
        for line in infile:
            line = line.strip()
            if not line.startswith(">"):  # Ignore headers
                full_sequence += line.upper()  

    return full_sequence
input_path = os.path.join("BEE_Data", "q2_data", "multiline_input.fasta")

full_sequence = load_fasta_sequences(input_path)
print(f"Loaded sequence length: {len(full_sequence)}")


Loaded sequence length: 12398


In [29]:
from collections import defaultdict

def build_first_order_markov_model(sequence):
    bases = 'ACGT'
    count = {b1: defaultdict(int) for b1 in bases}
    
    # Count the transitions between consecutive nucleotides
    for i in range(len(sequence) - 1):
        b1, b2 = sequence[i], sequence[i + 1]
        if b1 in bases and b2 in bases:
            count[b1][b2] += 1
    
    # Convert counts to probabilities
    transition_matrix = {}
    for b1 in bases:
        total = sum(count[b1][b2] for b2 in bases)
        transition_matrix[b1] = {}
        for b2 in bases:
            prob = count[b1][b2] / total if total > 0 else 0.0
            transition_matrix[b1][b2] = prob
    
    
    print("First-Order Transition Probability Matrix (rows: current base, columns: next base):\n")
    print("      A      C      G      T")

    print(f"A: {transition_matrix['A']['A']:.3f}  {transition_matrix['A']['C']:.3f}  {transition_matrix['A']['G']:.3f}  {transition_matrix['A']['T']:.3f}")
    print(f"C: {transition_matrix['C']['A']:.3f}  {transition_matrix['C']['C']:.3f}  {transition_matrix['C']['G']:.3f}  {transition_matrix['C']['T']:.3f}")
    print(f"G: {transition_matrix['G']['A']:.3f}  {transition_matrix['G']['C']:.3f}  {transition_matrix['G']['G']:.3f}  {transition_matrix['G']['T']:.3f}")
    print(f"T: {transition_matrix['T']['A']:.3f}  {transition_matrix['T']['C']:.3f}  {transition_matrix['T']['G']:.3f}  {transition_matrix['T']['T']:.3f}")

build_first_order_markov_model(full_sequence)


First-Order Transition Probability Matrix (rows: current base, columns: next base):

      A      C      G      T
A: 0.251  0.243  0.265  0.242
C: 0.247  0.254  0.245  0.254
G: 0.252  0.247  0.256  0.245
T: 0.271  0.237  0.250  0.241


<h3>Using 2nd order Markov Model</h3>

In [31]:
from collections import defaultdict

def build_second_order_markov_model(sequence):
    
    bases = 'ACGT'
    count = {b1b2: defaultdict(int) for b1b2 in [b1 + b2 for b1 in bases for b2 in bases]}
    
    # Count the transitions between pairs of consecutive nucleotides
    for i in range(len(sequence) - 2):
        b1b2, b3 = sequence[i:i+2], sequence[i + 2]
        if b1b2 in count and b3 in bases:
            count[b1b2][b3] += 1
    
    # Convert counts to probabilities
    transition_matrix = {}
    for b1b2 in count:
        total = sum(count[b1b2][b3] for b3 in bases)
        transition_matrix[b1b2] = {}
        for b3 in bases:
            prob = count[b1b2][b3] / total if total > 0 else 0.0
            transition_matrix[b1b2][b3] = prob
    
    # Print the transition matrix
    print("\nSecond-Order Transition Probability Matrix (rows: current pair of bases, columns: next base):\n")
    print("       A      C      G      T")
    
    
    
    for b1b2 in [b1 + b2 for b1 in bases for b2 in bases]:
        print(f"{b1b2}: {transition_matrix[b1b2]['A']:.3f}  {transition_matrix[b1b2]['C']:.3f}  {transition_matrix[b1b2]['G']:.3f}  {transition_matrix[b1b2]['T']:.3f}")


build_second_order_markov_model(full_sequence)



Second-Order Transition Probability Matrix (rows: current pair of bases, columns: next base):

       A      C      G      T
AA: 0.279  0.238  0.266  0.217
AC: 0.258  0.263  0.239  0.240
AG: 0.264  0.217  0.246  0.272
AT: 0.260  0.216  0.284  0.240
CA: 0.249  0.235  0.275  0.241
CC: 0.248  0.264  0.230  0.258
CG: 0.262  0.226  0.289  0.224
CT: 0.288  0.254  0.248  0.210
GA: 0.236  0.238  0.270  0.256
GC: 0.213  0.257  0.272  0.257
GG: 0.250  0.271  0.229  0.251
GT: 0.266  0.249  0.244  0.242
TA: 0.240  0.258  0.250  0.252
TC: 0.268  0.232  0.238  0.261
TG: 0.231  0.277  0.262  0.230
TT: 0.269  0.230  0.224  0.276
